In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import time

load_dotenv(r'C:\Users\Gurveer\ds-portfolio\.env')
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Model settings
PRIMARY_MODEL = "gpt-4o-mini"
FALLBACK_MODEL = "gpt-4o"
TEMPERATURE = 0.2

# Role instructions
son_role = (
    "You are the SON. You want to buy a motorcycle. "
    "State your reasons respectfully (independence, budget, safety training), "
    "answer questions, and be open to conditions (classes, gear, savings). "
    "Keep replies to 1–4 sentences. Always start your message with 'SON: '."
)

mother_role = (
    "You are the MOTHER. You care about safety and are currently against the purchase. "
    "Voice concerns calmly (injury risk, insurance, noise), ask clarifying questions, "
    "and suggest alternatives (postpone, safety course first, budget plan). "
    "Keep replies to 1–4 sentences. Always start your message with 'MOTHER: '."
)

father_role = (
    "You are the FATHER. You support the idea but want rules, a license and a clear budget. "
    "Help find a compromise (mandatory course, full gear, displacement limits, curfew, savings goals). "
    "Keep replies to 1–4 sentences. Always start your message with 'FATHER: '."
)

# Shared conversation history
conversation = [
    {"role": "system",
     "content": "This is a respectful family discussion. Keep it constructive, specific, and concise."}
]

def speak(role_instruction: str, conversation: list, model: str = PRIMARY_MODEL) -> str:
    try:
        resp = client.chat.completions.create(
            model=model,
            temperature=TEMPERATURE,
            messages=[{"role": "system", "content": role_instruction}] + conversation
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        print(f"[WARN] {type(e).__name__}: {e} — retrying with '{FALLBACK_MODEL}'")
        time.sleep(0.5)
        resp = client.chat.completions.create(
            model=FALLBACK_MODEL,
            temperature=TEMPERATURE,
            messages=[{"role": "system", "content": role_instruction}] + conversation
        )
        return resp.choices[0].message.content.strip()

# Seed the conversation (Son starts)
first_message = (
    "SON: I'd like to buy a small motorcycle. I've been researching safety courses "
    "and budgeting for gear, and I think I can handle it responsibly."
)
conversation.append({"role": "user", "content": first_message})
print(first_message)

# Turn order: Mother → Father → Son
turn_order = [
    ("MOTHER", mother_role),
    ("FATHER", father_role),
    ("SON", son_role),
]

ROUNDS = 5

for r in range(ROUNDS):
    for speaker_name, role_prompt in turn_order:
        reply = speak(role_prompt, conversation)
        if not reply.upper().startswith(f"{speaker_name}:"):
            reply = f"{speaker_name}: " + reply
        print(reply)
        conversation.append({"role": "assistant", "content": reply})

# Wrap-up
wrap = speak(father_role, conversation)
if not wrap.upper().startswith("FATHER:"):
    wrap = "FATHER: " + wrap
print(wrap)
conversation.append({"role": "assistant", "content": wrap})

SON: I’d like to buy a small motorcycle. I’ve been researching safety courses and budgeting for gear, and I think I can handle it responsibly.
MOTHER: I appreciate your research and responsibility, but I have concerns about the injury risk associated with motorcycles. Have you considered how we would handle insurance costs and potential noise issues in our neighborhood? Perhaps we could postpone the purchase until you complete a safety course and create a more detailed budget plan.
FATHER: I support your interest in getting a motorcycle, but we need to set some clear rules first. You’ll need to complete a mandatory safety course and wear full gear at all times. Let’s also establish a budget for the motorcycle and gear, set a displacement limit for the bike, and agree on a curfew for when you can ride. Additionally, I’d like you to save a certain amount each month to contribute to the purchase. How does that sound?
SON: I understand your concerns, and I’m willing to take a safety course